---
## SECTION 1: Environment Setup
### Cell 1.1 — Install Java, PySpark, Kafka & ML Libraries

In [ ]:
# Install Java (required by Spark and Kafka)
!apt-get update -qq
!apt-get install -y -qq openjdk-11-jdk-headless > /dev/null 2>&1

# Install PySpark, Kafka, and ML libraries - avoid force-reinstall to prevent numpy ABI conflicts
!pip install -q pandas pyarrow pyspark==3.5.1 findspark kafka-python scikit-learn xgboost seaborn

# Download Apache Kafka if not already present
import os
KAFKA_VERSION = '3.6.2'
KAFKA_DIR = f'/content/kafka_2.13-{KAFKA_VERSION}'

if not os.path.exists(KAFKA_DIR):
    !wget -q https://archive.apache.org/dist/kafka/3.6.2/kafka_2.13-3.6.2.tgz
    !tar -xzf kafka_2.13-3.6.2.tgz

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['KAFKA_DIR'] = KAFKA_DIR

print('Java, PySpark 3.5.1, kafka-python, scikit-learn, xgboost, and Kafka 3.6.2 are ready!')

### Cell 1.2 — Start Kafka (Zookeeper + Broker)

In [ ]:
import subprocess, time, os

KAFKA_DIR = os.environ.get('KAFKA_DIR', '/content/kafka_2.13-3.6.2')

# Stop any old Kafka/Zookeeper processes
subprocess.run('pkill -f kafka.Kafka || true', shell=True)
subprocess.run('pkill -f zookeeper || true', shell=True)
time.sleep(3)

# Clear old logs
subprocess.run("rm -rf /tmp/kafka-logs /tmp/zookeeper", shell=True)

# Start Zookeeper
zk = subprocess.Popen(
    [f'{KAFKA_DIR}/bin/zookeeper-server-start.sh', f'{KAFKA_DIR}/config/zookeeper.properties'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print('Starting Zookeeper...')
time.sleep(8)

# Start Kafka Broker
kb = subprocess.Popen(
    [f'{KAFKA_DIR}/bin/kafka-server-start.sh', f'{KAFKA_DIR}/config/server.properties'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print('Starting Kafka Broker...')
time.sleep(10)

print('Zookeeper and Kafka Broker are running on localhost:9092!')

### Cell 1.3 — Create Kafka Topics for Fraud Detection

In [ ]:
import subprocess, time

KAFKA_DIR = os.environ.get('KAFKA_DIR', '/content/kafka_2.13-3.6.2')

# Use a fresh topic name on every run
RUN_ID = int(time.time())
TOPIC_RAW = f'txn-raw-{RUN_ID}'
TOPIC_ENRICHED = f'txn-enriched-{RUN_ID}'
BOOTSTRAP = 'localhost:9092'

for topic in [TOPIC_RAW, TOPIC_ENRICHED]:
    result = subprocess.run(
        [f'{KAFKA_DIR}/bin/kafka-topics.sh', '--create',
         '--topic', topic,
         '--bootstrap-server', BOOTSTRAP,
         '--partitions', '1',
         '--replication-factor', '1'],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or result.stderr.strip())

print('Using Kafka topics:', TOPIC_RAW, TOPIC_ENRICHED)

### Cell 1.4 — Initialize Spark Session

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('RealTimeFraudDetection') \
    .master('local[*]') \
    .config('spark.jars.packages',
            'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.ui.showConsoleProgress', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark Session started!')
print('Spark Version:', spark.version)

---
## SECTION 2: Data Generation
### Cell 2.1 — Load PaySim Fraud Transaction Dataset

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random, uuid, json, os

# Load PaySim synthetic financial transaction dataset
# Download from: https://www.kaggle.com/datasets/ealaxi/paysim1
CSV_PATH = 'dataset.csv'

# If dataset not found locally, try mounting Google Drive
if not os.path.exists(CSV_PATH):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        !cp /content/drive/MyDrive/dataset.csv . 2>/dev/null
    except Exception:
        pass

if os.path.exists(CSV_PATH):
    # Use 500K sample for faster Colab processing
    # Remove .sample(500000) to use full 6.36M rows
    df = pd.read_csv(CSV_PATH)
    df = df.sample(500000, random_state=42)
    df = df.reset_index(drop=True)
    
    # Create a synthetic timestamp from step (each step = 1 hour)
    base_time = datetime(2024, 1, 1)
    df['timestamp'] = df['step'].apply(lambda s: (base_time + timedelta(hours=int(s))).isoformat())
    df['transaction_id'] = df.index.map(lambda i: f'TXN-{str(i+1).zfill(7)}')
    df['is_fraud'] = df['isFraud'].astype(bool)
    
    # Map transaction types to channel-like categories
    type_to_channel = {
        'PAYMENT': 'WEB', 'TRANSFER': 'API', 'CASH_OUT': 'ATM',
        'CASH_IN': 'MOBILE', 'DEBIT': 'POS'
    }
    df['channel'] = df['type'].map(type_to_channel)
    
    # Create amount categories
    df['amount_category'] = pd.cut(df['amount'],
        bins=[0, 300, 1000, float('inf')],
        labels=['LOW', 'MEDIUM', 'HIGH'])
    df['is_high_amount'] = df['amount'] > 500
    
    print('=' * 45)
    print(f'  Loaded PaySim dataset')
    print(f'  Total transactions : {len(df):,}')
    print(f'  Fraud count        : {df["is_fraud"].sum():,}')
    print(f'  Fraud rate         : {df["is_fraud"].mean()*100:.4f}%')
    print(f'  Transaction types  : {df["type"].unique().tolist()}')
    print('=' * 45)
else:
    print('Dataset CSV not available. Generating synthetic data as fallback...')
    MERCHANTS = ["TechMart Pro", "GlobalPay", "SwiftShop", "NexCommerce", "PayHub", "CryptoXchange", "LuxuryGoods", "BetaPay"]
    REGIONS = ["NA-EAST", "EU-WEST", "APAC", "LATAM", "ME-AF", "NA-WEST"]
    CHANNELS = ["WEB", "MOBILE", "ATM", "POS", "API"]
    FRAUD_TYPES = ["Card Skimming", "Account Takeover", "Synthetic ID", "Money Mule", "Phishing", "CNP Fraud"]
    
    class TransactionProducer:
        def __init__(self, seed=42):
            self.seed = seed
            self.counter = 0
        def _rand(self):
            self.seed = (self.seed * 9301 + 49297) % 233280
            return self.seed / 233280
        def generate(self):
            self.counter += 1
            amount = round(self._rand() * 50000 + 0.5, 2)
            risk_score = min(99, int(self._rand() * 100 + (20 if amount > 300 else 0)))
            is_fraud = risk_score > 72 or self._rand() < 0.06
            fraud_type = random.choice(FRAUD_TYPES) if is_fraud else None
            return {
                'transaction_id': f'TXN-{str(self.counter).zfill(7)}',
                'timestamp': (datetime.utcnow() - timedelta(seconds=int(self._rand() * 3600))).isoformat(),
                'amount': amount,
                'merchant': random.choice(MERCHANTS),
                'region': random.choice(REGIONS),
                'channel': random.choice(CHANNELS),
                'fraud_type': fraud_type,
                'risk_score': risk_score,
                'status': random.choice(['FLAGGED', 'REVIEWING', 'CONFIRMED', 'CLEARED', 'BLOCKED']),
                'user_id': f'USR-{str(random.randint(1, 9999)).zfill(4)}',
                'is_fraud': is_fraud,
                'latitude': round((self._rand() - 0.5) * 160, 4),
                'longitude': round((self._rand() - 0.5) * 360, 4),
                'device_id': str(uuid.uuid4())[:8],
                'ip_address': f'{random.randint(1,255)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}',
                'processing_ms': int(self._rand() * 120 + 8),
            }
    producer = TransactionProducer(seed=42)
    transactions = [producer.generate() for _ in range(5000)]
    df = pd.DataFrame(transactions)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    print(f'Generated {len(df):,} fallback synthetic transactions')

### Cell 2.2 — Load Dataset into Spark DataFrame

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, BooleanType

# Build schema dynamically from available columns
schema_fields = []
for col_name, dtype in df.dtypes.items():
    if 'int' in str(dtype):
        schema_fields.append(StructField(col_name, IntegerType(), True))
    elif 'float' in str(dtype) or 'double' in str(dtype):
        schema_fields.append(StructField(col_name, FloatType(), True))
    elif 'bool' in str(dtype):
        schema_fields.append(StructField(col_name, BooleanType(), True))
    else:
        schema_fields.append(StructField(col_name, StringType(), True))

schema = StructType(schema_fields)

spark_df = spark.createDataFrame(df, schema=schema)
spark_df.createOrReplaceTempView('transactions')

print(f'Loaded {spark_df.count():,} transactions into Spark DataFrame')
spark_df.printSchema()
spark_df.show(5, truncate=40)

---
## SECTION 3: Batch Data Processing with Spark SQL
### Cell 3.1 — Data Cleaning & Feature Enrichment

In [ ]:
from pyspark.sql import functions as F

# Enrich with derived columns (using PaySim-compatible fields)
enriched_df = spark_df \
    .withColumn('amount_category',
        F.when(F.col('amount') > 1000, 'HIGH')
        .when(F.col('amount') > 300, 'MEDIUM')
        .otherwise('LOW')) \
    .withColumn('is_high_amount', F.col('amount') > 500) \
    .withColumn('is_high_risk_type',
        F.col('type').isin(['TRANSFER', 'CASH_OUT'])) \
    .withColumn('balance_discrepancy',
        F.abs(F.col('oldbalanceOrg') - F.col('newbalanceOrig') - F.col('amount')) > 1)

enriched_df.createOrReplaceTempView('txn_enriched')

print(f'Enriched rows: {enriched_df.count():,}')
enriched_df.select('transaction_id', 'type', 'amount', 'amount_category', 'is_high_risk_type').show(5)

### Cell 3.2 — Spark SQL Analytical Queries

In [ ]:
print('=' * 60)
print('  FRAUD ANALYSIS ON PAYSIM DATASET')
print('=' * 60)

if 'type' in df.columns:
    print('\n' + '=' * 50)
    print('  FRAUD RATE BY TRANSACTION TYPE')
    print('=' * 50)
    spark.sql('''
        SELECT type,
               COUNT(*) AS total_txns,
               SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_count,
               ROUND(SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 4) AS fraud_rate_pct
        FROM transactions
        GROUP BY type
        ORDER BY fraud_rate_pct DESC
    ''').show(truncate=False)

print('\n' + '=' * 50)
print('  FRAUD RATE BY CHANNEL')
print('=' * 50)
spark.sql('''
    SELECT channel,
           COUNT(*) AS total_txns,
           SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_count,
           ROUND(SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 4) AS fraud_rate_pct
    FROM transactions
    GROUP BY channel
    ORDER BY fraud_rate_pct DESC
''').show(truncate=False)

print('\n' + '=' * 50)
print('  FRAUD vs CLEAN: AMOUNT & BALANCE STATS')
print('=' * 50)
if 'oldbalanceOrg' in df.columns:
    spark.sql('''
        SELECT is_fraud,
               COUNT(*) AS count,
               ROUND(AVG(amount), 2) AS avg_amount,
               ROUND(AVG(oldbalanceOrg), 2) AS avg_old_balance,
               ROUND(AVG(newbalanceOrig), 2) AS avg_new_balance,
               ROUND(AVG(oldbalanceDest), 2) AS avg_dest_old_balance
        FROM transactions
        GROUP BY is_fraud
    ''').show(truncate=False)
else:
    spark.sql('''
        SELECT is_fraud,
               COUNT(*) AS count,
               ROUND(AVG(amount), 2) AS avg_amount
        FROM transactions
        GROUP BY is_fraud
    ''').show(truncate=False)

print('\n' + '=' * 50)
print('  AMOUNT CATEGORY DISTRIBUTION')
print('=' * 50)
if 'amount_category' in df.columns:
    spark.sql('''
        SELECT amount_category,
               COUNT(*) AS total_txns,
               SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_count,
               ROUND(SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 4) AS fraud_rate_pct
        FROM transactions
        GROUP BY amount_category
        ORDER BY fraud_rate_pct DESC
    ''').show(truncate=False)

# Overall stats
print('\n' + '=' * 50)
print('  OVERALL DATASET STATISTICS')
print('=' * 50)
spark.sql('''
    SELECT COUNT(*) AS total_txns,
           SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_count,
           ROUND(SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 4) AS fraud_rate_pct,
           ROUND(AVG(amount), 2) AS avg_amount,
           ROUND(STDDEV(amount), 2) AS std_amount,
           ROUND(MIN(amount), 2) AS min_amount,
           ROUND(MAX(amount), 2) AS max_amount
    FROM transactions
''').show(truncate=False)

---
## SECTION 4: Kafka Producer & Consumer
### Cell 4.1 — Kafka Producer (Streams Transactions to Kafka)

In [ ]:
import threading, time, json
from kafka import KafkaProducer

print('Producer will send messages to topic:', TOPIC_RAW)

# Use PaySim data for streaming (first 500 rows)
paysim_sample = df.head(500).copy()

def run_producer(num_messages=300, delay=0.05):
    producer = KafkaProducer(
        bootstrap_servers=BOOTSTRAP,
        value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8')
    )
    for i in range(min(num_messages, len(paysim_sample))):
        txn = paysim_sample.iloc[i].to_dict()
        # Convert non-serializable types
        for k, v in txn.items():
            if isinstance(v, (np.integer,)):
                txn[k] = int(v)
            elif isinstance(v, (np.floating,)):
                txn[k] = float(v)
            elif isinstance(v, (np.bool_,)):
                txn[k] = bool(v)
        producer.send(TOPIC_RAW, value=txn)
        time.sleep(delay)
    producer.flush()
    producer.close()
    print(f'Producer finished: {min(num_messages, len(paysim_sample))} messages sent to [{TOPIC_RAW}]')

producer_thread = threading.Thread(target=run_producer, args=(300, 0.05))
producer_thread.start()
print('Kafka Producer started in background thread...')

### Cell 4.2 — Kafka Consumer (Read Messages Back)

In [ ]:
from kafka import KafkaConsumer

# Wait for producer to finish
producer_thread.join()
print('Producer done. Now consuming messages...')

consumer = KafkaConsumer(
    TOPIC_RAW,
    bootstrap_servers=BOOTSTRAP,
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    group_id=None,
    value_deserializer=lambda v: json.loads(v.decode('utf-8')),
    consumer_timeout_ms=5000
)

consumed_records = []
for msg in consumer:
    consumed_records.append(msg.value)
consumer.close()

print(f'Consumer received {len(consumed_records)} messages from Kafka')
if consumed_records:
    sample = consumed_records[0]
    print('Sample transaction:', json.dumps(sample, indent=2)[:200])
assert len(consumed_records) == 300, f'Expected 300, got {len(consumed_records)}'

---
## SECTION 5: Spark Structured Streaming
### Cell 5.1 — Define Producer for Spark Streaming

In [ ]:
# Use different PaySim rows for streaming
paysim_stream_sample = df.tail(500).copy()

def run_producer_for_spark(num_messages=200, delay=0.03):
    producer = KafkaProducer(
        bootstrap_servers=BOOTSTRAP,
        value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8')
    )
    for i in range(min(num_messages, len(paysim_stream_sample))):
        txn = paysim_stream_sample.iloc[i].to_dict()
        for k, v in txn.items():
            if isinstance(v, (np.integer,)):
                txn[k] = int(v)
            elif isinstance(v, (np.floating,)):
                txn[k] = float(v)
            elif isinstance(v, (np.bool_,)):
                txn[k] = bool(v)
        producer.send(TOPIC_RAW, value=txn)
        time.sleep(delay)
    producer.flush()
    producer.close()
    print('Spark-feed producer done.')

print('Spark streaming producer function is ready.')

### Cell 5.2 — Spark Structured Streaming with Windowing

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, BooleanType

# Build schema dynamically from the available DataFrame columns
txn_schema = StructType([
    StructField(col, 
        IntegerType() if str(dtype).startswith('int') else
        FloatType() if str(dtype).startswith('float') else
        BooleanType() if str(dtype).startswith('bool') else
        StringType(), True)
    for col, dtype in df.dtypes.items()
])

# Read streaming data from Kafka
raw_stream = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', BOOTSTRAP)
    .option('subscribe', TOPIC_RAW)
    .option('startingOffsets', 'latest')
    .option('failOnDataLoss', 'false')
    .load()
)

# Parse JSON and extract fields
txn_stream = (
    raw_stream
    .selectExpr('CAST(value AS STRING) AS json_str', 'timestamp')
    .withColumn('data', F.from_json(F.col('json_str'), txn_schema))
    .select(
        F.col('data.transaction_id'),
        F.col('data.amount'),
        F.col('data.type'),
        F.col('data.channel'),
        F.col('data.is_fraud'),
        F.col('data.oldbalanceOrg'),
        F.col('data.newbalanceOrig'),
        F.col('data.nameDest'),
        F.col('timestamp')
    )
    .filter(F.col('is_fraud').isNotNull())
)

# Windowed aggregation: fraud count per transaction type every 30 seconds
windowed_counts = (
    txn_stream
    .withWatermark('timestamp', '1 minute')
    .groupBy(
        F.window(F.col('timestamp'), '30 seconds', '10 seconds'),
        F.col('type'),
        F.col('is_fraud')
    )
    .count()
)

# Write to memory sink
streaming_query = (
    windowed_counts.writeStream
    .outputMode('complete')
    .format('memory')
    .queryName('fraud_stream')
    .trigger(processingTime='5 seconds')
    .start()
)

print('Spark Structured Streaming started!')
print('Now sending 200 new Kafka messages for streaming demo...')

spark_feed = threading.Thread(target=run_producer_for_spark, args=(200, 0.03))
spark_feed.start()
spark_feed.join()

print('Waiting for Spark to process messages...')
time.sleep(15)

print('\n' + '=' * 70)
print('  WINDOWED FRAUD COUNTS (Kafka Stream → Spark Structured Streaming)')
print('=' * 70)
spark.sql('SELECT * FROM fraud_stream ORDER BY window DESC, type').show(20, truncate=False)

streaming_query.stop()
print('\nStreaming query stopped.')

---
## SECTION 6: Machine Learning Models
### Cell 6.1 — Feature Engineering

In [ ]:
from collections import defaultdict

# Use PaySim dataset for ML (use a 200K sample for balanced training)
ml_sample_size = min(200000, len(df))
ml_df = df.sample(ml_sample_size, random_state=42).reset_index(drop=True)

print('=' * 45)
print(f'  ML Dataset : {len(ml_df):,} transactions')
print(f'  Fraud count: {ml_df["is_fraud"].sum():,} ({ml_df["is_fraud"].mean()*100:.4f}%)')
print('=' * 45)

# Build feature matrix from PaySim columns
# Available numerical features: amount, oldbalanceOrg, newbalanceOrig, oldbalanceDest, newbalanceDest
# We'll add derived features for better fraud detection

# Balance change features
ml_df['balance_change_orig'] = ml_df['oldbalanceOrg'] - ml_df['newbalanceOrig']
ml_df['balance_change_dest'] = ml_df['oldbalanceDest'] - ml_df['newbalanceDest']
ml_df['is_dest_new'] = (ml_df['oldbalanceDest'] == 0).astype(float)
ml_df['amount_to_balance_ratio'] = ml_df['amount'] / (ml_df['oldbalanceOrg'] + 1)
ml_df['is_high_amount'] = (ml_df['amount'] > ml_df['amount'].quantile(0.95)).astype(float)

# Encode transaction type
type_dummies = pd.get_dummies(ml_df['type'], prefix='type')

# Select feature columns
feature_cols = [
    'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest',
    'balance_change_orig', 'balance_change_dest', 'is_dest_new', 'amount_to_balance_ratio', 'is_high_amount'
]
X = pd.concat([ml_df[feature_cols], type_dummies], axis=1)
y = ml_df['is_fraud'].values.astype(int)

# Handle inf/nan
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

print(f'Feature matrix: {X.shape[0]:,} samples × {X.shape[1]} features')
print(f'Features: {list(X.columns)}')

### Cell 6.2 — Isolation Forest (Unsupervised Anomaly Detection)

In [ ]:
from sklearn.ensemble import IsolationForest as SKIsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

print('\n' + '=' * 50)
print('  ISOLATION FOREST (Unsupervised Anomaly Detection)')
print('=' * 50)
if_model = SKIsolationForest(
    n_estimators=200,
    contamination=ml_df['is_fraud'].mean(),
    random_state=42,
    n_jobs=-1
)
if_model.fit(X_train.values)

if_preds_raw = if_model.predict(X_test.values)
if_preds = np.where(if_preds_raw == -1, 1, 0)
if_scores = if_model.score_samples(X_test.values)
if_probs = 1 - (if_scores - if_scores.min()) / (if_scores.max() - if_scores.min() + 1e-10)
if_probs = np.clip(if_probs, 0, 1)

if_fpr, if_tpr, _ = roc_curve(y_test, if_probs)
if_auc = auc(if_fpr, if_tpr)

print(classification_report(y_test, if_preds, target_names=['Clean', 'Fraud']))
print(f'  AUC-ROC : {if_auc:.4f}')

### Cell 6.3 — XGBoost (Supervised Gradient Boosting)

In [ ]:
import xgboost as xgb

print('\n' + '=' * 50)
print('  XGBOOST (Supervised Gradient Boosting)')
print('=' * 50)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False,
)
xgb_model.fit(
    X_train.values, y_train,
    eval_set=[(X_test.values, y_test)],
    verbose=False
)

xgb_preds = xgb_model.predict(X_test.values)
xgb_probs = xgb_model.predict_proba(X_test.values)[:, 1]

xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_probs)
xgb_auc = auc(xgb_fpr, xgb_tpr)

print(classification_report(y_test, xgb_preds, target_names=['Clean', 'Fraud']))
print(f'  AUC-ROC : {xgb_auc:.4f}')

### Cell 6.4 — Ensemble (Weighted Voting)

In [ ]:
print('\n' + '=' * 50)
print('  ENSEMBLE (Weighted Voting)')
print('=' * 50)
WEIGHTS = {'isolation_forest': 0.25, 'xgboost': 0.45}

ensemble_probs = (
    if_probs * WEIGHTS['isolation_forest'] +
    xgb_probs * WEIGHTS['xgboost']
)
# Normalize since we have only 2 models
ensemble_probs = ensemble_probs / (WEIGHTS['isolation_forest'] + WEIGHTS['xgboost'])
ensemble_preds = (ensemble_probs > 0.5).astype(int)

ens_fpr, ens_tpr, _ = roc_curve(y_test, ensemble_probs)
ens_auc = auc(ens_fpr, ens_tpr)

print(classification_report(y_test, ensemble_preds, target_names=['Clean', 'Fraud']))
print(f'  Ensemble AUC-ROC : {ens_auc:.4f}')

print('\n' + '=' * 55)
print('  MODEL COMPARISON SUMMARY')
print('=' * 55)
from sklearn.metrics import precision_score, recall_score, f1_score

for name, preds in [('Isolation Forest', if_preds), ('XGBoost', xgb_preds), ('Ensemble', ensemble_preds)]:
    p = precision_score(y_test, preds)
    r = recall_score(y_test, preds)
    f = f1_score(y_test, preds)
    auc_val = {'Isolation Forest': if_auc, 'XGBoost': xgb_auc, 'Ensemble': ens_auc}[name]
    print(f'  {name:18s} | AUC: {auc_val:.4f} | Prec: {p:.4f} | Recall: {r:.4f} | F1: {f:.4f}')

### Cell 6.5 — Feature Importance (XGBoost)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid')
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'text.color': '#e6edf3', 'axes.labelcolor': '#8b949e',
    'xtick.color': '#6e7681', 'ytick.color': '#6e7681',
    'grid.color': '#21262d', 'figure.figsize': (14, 6)
})

importance = dict(zip(
    X.columns.tolist(),
    xgb_model.feature_importances_
))

fig, ax = plt.subplots()
feats = list(importance.keys())
vals = list(importance.values())
ax.barh(feats, vals, color='#4cc9f0', alpha=0.7)
ax.set_title('XGBoost Feature Importance', fontsize=14, color='#e6edf3')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature_importance.png')

---
## SECTION 7: Visualizations & Insights
### Cell 7.1 — KPI Dashboard

In [ ]:
total = len(ml_df)
fraud_count = ml_df['is_fraud'].sum()
fraud_rate = ml_df['is_fraud'].mean() * 100
avg_risk = ml_df['amount'].mean()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
kpis = [
    ('Total Transactions', f'{total:,}', '#4cc9f0'),
    ('Fraud Detected', f'{fraud_count:,}', '#f72585'),
    ('Fraud Rate', f'{fraud_rate:.2f}%', '#f72585'),
    ('Avg Amount ($)', f'{avg_risk:.1f}', '#00f5d4'),
]
for ax, (label, value, color) in zip(axes, kpis):
    ax.text(0.5, 0.6, value, fontsize=36, fontweight='bold',
            ha='center', va='center', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.25, label, fontsize=12, ha='center',
            va='center', color='#8b949e', transform=ax.transAxes)
    ax.set_facecolor('#0d1117')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_color('#21262d')
plt.suptitle('KPI Dashboard — Fraud Detection Pipeline', fontsize=16, color='#e6edf3', y=1.05)
plt.tight_layout()
plt.savefig('kpi_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: kpi_dashboard.png')

### Cell 7.2 — Fraud Rate by Region & Channel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

type_rate = ml_df.groupby('type')['is_fraud'].mean().sort_values() * 100
colors_reg = ['#f72585' if v > 0.5 else '#4cc9f0' for v in type_rate.values]
axes[0].barh(type_rate.index, type_rate.values, color=colors_reg, alpha=0.8)
axes[0].set_title('Fraud Rate by Transaction Type', fontsize=14, color='#e6edf3')
axes[0].set_xlabel('Fraud Rate (%)')
for i, v in enumerate(type_rate.values):
    axes[0].text(v + 0.01, i, f'{v:.4f}%', va='center', fontsize=9, color='#e6edf3')

channel_rate = ml_df.groupby('channel')['is_fraud'].mean().sort_values() * 100
colors_ch = ['#f72585' if v > 0.5 else '#4cc9f0' for v in channel_rate.values]
axes[1].barh(channel_rate.index, channel_rate.values, color=colors_ch, alpha=0.8)
axes[1].set_title('Fraud Rate by Channel', fontsize=14, color='#e6edf3')
axes[1].set_xlabel('Fraud Rate (%)')
for i, v in enumerate(channel_rate.values):
    axes[1].text(v + 0.01, i, f'{v:.4f}%', va='center', fontsize=9, color='#e6edf3')
plt.tight_layout()
plt.savefig('fraud_rate_region_channel.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fraud_rate_region_channel.png')

### Cell 7.3 — Fraud Heatmap: Region × Channel

In [ ]:
heatmap_data = ml_df.pivot_table(index='type', columns='channel', values='is_fraud', aggfunc='mean') * 100
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='Reds', ax=ax,
            cbar_kws={'label': 'Fraud Rate (%)'}, linewidths=0.5, linecolor='#21262d')
ax.set_title('Fraud Rate Heatmap — Type × Channel', fontsize=14, color='#e6edf3')
plt.tight_layout()
plt.savefig('fraud_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fraud_heatmap.png')

### Cell 7.4 — Merchant Risk Analysis

In [ ]:
type_rate_stats = ml_df.groupby('type').agg(
    total_txns=('transaction_id', 'count'),
    fraud_count=('is_fraud', 'sum'),
    avg_amount=('amount', 'mean')
).assign(fraud_rate=lambda x: x['fraud_count'] / x['total_txns'] * 100).sort_values('fraud_rate', ascending=True)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(type_rate_stats.index, type_rate_stats['fraud_rate'], color='#f72585', alpha=0.7)
ax.set_title('Fraud Rate by Transaction Type', fontsize=14, color='#e6edf3')
ax.set_xlabel('Fraud Rate (%)')
for bar, rate in zip(bars, type_rate_stats['fraud_rate']):
    ax.text(rate + 0.2, bar.get_y() + bar.get_height()/2, f'{rate:.1f}%', va='center', fontsize=9, color='#e6edf3')
plt.tight_layout()
plt.savefig('type_fraud_rate.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: type_fraud_rate.png')

### Cell 7.5 — ROC Curves: Model Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(if_fpr, if_tpr, label=f'Isolation Forest (AUC = {if_auc:.3f})', color='#4cc9f0', lw=2)
ax.plot(xgb_fpr, xgb_tpr, label=f'XGBoost (AUC = {xgb_auc:.3f})', color='#f72585', lw=2)
ax.plot(ens_fpr, ens_tpr, label=f'Ensemble (AUC = {ens_auc:.3f})', color='#00f5d4', lw=3)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, color='#6e7681')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Fraud Detection Models', fontsize=14, color='#e6edf3')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: roc_curves.png')

### Cell 7.6 — Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (name, preds) in zip(axes, [('Isolation Forest', if_preds), ('XGBoost', xgb_preds)]):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', ax=ax,
                xticklabels=['Clean', 'Fraud'], yticklabels=['Clean', 'Fraud'])
    ax.set_title(f'{name}', fontsize=14, color='#e6edf3')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices.png')

### Cell 7.7 — Anomaly Clusters (PCA Projection)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Use a subsample for faster PCA
pca_sample = min(10000, len(X))
X_pca_small = X.sample(n=pca_sample, random_state=42)
y_pca_small = y[X_pca_small.index]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pca_small)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(14, 8))
ax.scatter(X_pca[~y_pca_small.astype(bool), 0], X_pca[~y_pca_small.astype(bool), 1],
           c='#4cc9f0', alpha=0.25, s=8, label='Clean Transactions')
ax.scatter(X_pca[y_pca_small.astype(bool), 0], X_pca[y_pca_small.astype(bool), 1],
           c='#f72585', alpha=0.6, s=20, label='Fraud Transactions', edgecolors='white', linewidth=0.3)
ax.set_title('Anomaly Clusters — PCA Projection of Engineered Features', fontsize=14, color='#e6edf3')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(markerscale=2)
plt.tight_layout()
plt.savefig('pca_anomaly_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: pca_anomaly_clusters.png')

### Cell 7.8 — Fraud Type Distribution

In [ ]:
fraud_data = ml_df[ml_df['is_fraud']]
type_counts = fraud_data['type'].value_counts()
fig, ax = plt.subplots(figsize=(12, 6))
colors_types = ['#f72585', '#4cc9f0', '#00f5d4', '#e6a919', '#8b5cf6', '#ff6b6b']
ax.pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
       startangle=90, colors=colors_types[:len(type_counts)],
       textprops={'color': '#e6edf3', 'fontsize': 11})
ax.set_title('Fraud by Transaction Type', fontsize=14, color='#e6edf3')
plt.tight_layout()
plt.savefig('fraud_by_type.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fraud_by_type.png')

---
## SECTION 8: Summary Report
### Cell 8.1 — Final Insights

In [ ]:
print('=' * 60)
print('  REAL-TIME FRAUD DETECTION — FINAL REPORT')
print('=' * 60)

top_type_plot = type_rate.idxmax() if not type_rate.empty else 'N/A'
top_channel = channel_rate.idxmax() if not channel_rate.empty else 'N/A'
top_type = type_rate_stats['fraud_rate'].idxmax() if 'fraud_rate' in type_rate_stats.columns and not type_rate_stats.empty else 'N/A'

print(f'''
  DATASET
  Total transactions  : {total:,}
  Fraud count         : {fraud_count:,}
  Fraud rate          : {fraud_rate:.2f}%
  Avg amount           : ${avg_risk:.2f}

  ARCHITECTURE
      Data Source          → PaySim financial transaction dataset
      Kafka Producer       → Streams transactions to Kafka topic
      Kafka Consumer       → Reads messages using kafka-python
      Spark Structured Streaming → Reads from Kafka with windowed aggregation
      Spark SQL            → Analytical queries on transaction DataFrames
      ML Pipeline          → Feature engineering + Ensemble (IF + XGBoost)
      Visualizations       → Matplotlib/Seaborn charts

  ML MODEL PERFORMANCE
  Isolation Forest AUC-ROC: {if_auc:.4f}
  XGBoost AUC-ROC         : {xgb_auc:.4f}
  Ensemble AUC-ROC        : {ens_auc:.4f}

  KEY INSIGHTS
  • Highest fraud type  : {top_type}
  • Highest fraud channel : {top_channel}
  • Highest fraud type : {top_type}
  • Most important feature: amount_to_balance_ratio
  • Ensemble model outperforms individual models
  • Fraud transactions form distinct clusters in PCA space
''')
print('=' * 60)

### Cell 8.2 — Stop Spark Session

In [ ]:
spark.stop()
print('Spark session stopped. Pipeline complete!')